# GC 1-Minute – Barrier Strategy with Gradient Boosting – Session Structure

Version with **multi-day session context** (previous session, night session, US opening range) and only the indicator
periods that mattered in the period test. Results go to `models_barrier_session/` and `..._barrier_session_signals.csv`,
so earlier runs are not overwritten.

Instead of forecasting the price, this notebook learns the trading rule directly. For every 1-minute candle and every setup:

> **Long:** is the price `+target` ticks above the entry (close of the candle) **before** it is `stop` ticks below,
> within `horizon` minutes? – **Short:** the same mirrored.

A target reached in minute 13 of a 15-minute window counts as a win. Two classifiers per setup (long, short) estimate these
probabilities; a trade is taken only when a probability is above a threshold that is chosen on the **validation period**
and then applied **once** to the separate test CSV.

**Inputs:** only the CSV columns Open, Close, High, Low, Ask, Bid, Delta, Delta/Volume, Session Delta, Session Delta %,
Max Delta, Min Delta, Delta Change, Volume – plus the time of day / weekday from the timestamp (`USE_TIME_FEATURES`).
All other CSV columns are ignored. The timestamp and all original columns stay in the output CSV.

**Time split without overlap:** training data = everything in the training CSV **before** the first candle of the test CSV.
Of that, the last `VALIDATION_FRACTION` is validation (threshold selection); the test CSV is only used at the very end.

**Comparison:** on the test data the model is compared with *random direction* and *always long* using the same entry
times and the same exit rules. The model is only useful if it is clearly better than both **after costs**.

## 0. Configuration

In [1]:
from pathlib import Path

SYMBOL = "GC"
TRAIN_CSV_PATH = Path(r"E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv")
TEST_CSV_PATH = Path(r"E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv")
TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH.with_name(TEST_CSV_PATH.stem + "_barrier_session_signals.csv")
MODEL_DIR = Path("models_barrier_session")   # separate folder -> earlier runs stay untouched
LOG_PATH = MODEL_DIR / "barrier_log.csv"

# Model inputs: ONLY these CSV columns (+ time features from "Time UTC" if USE_TIME_FEATURES)
INPUT_COLUMNS = [
    "Open", "Close", "High", "Low", "Ask", "Bid", "Delta", "Delta/Volume",
    "Session Delta", "Session Delta %", "Max Delta", "Min Delta", "Delta Change", "Volume",
]
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]   # exported as text "59.15%" -> 0.5915
USE_TIME_FEATURES = True                                 # hour of day + weekday from the timestamp

# Additional indicators (calculated only from the input columns above)
# Only the periods that carried information in the period test (short RSI and 5-30 minute averages removed)
RSI_PERIODS = [100]                    # RSI (Wilder) on 1-minute closes
TWAP_WINDOWS = [60, 120, 240]          # rolling TWAP in minutes (+ session TWAP, reset after every pause)
VWAP_WINDOWS = [60, 120, 240]          # rolling VWAP in minutes (+ session VWAP, reset after every pause)

# Session structure (New York time, daylight saving time handled automatically)
SESSION_START_HOUR_ET = 18      # CME Globex trading day: 18:00 -> 17:00 New York time (daily break 17:00-18:00)
US_OPEN_ET = (9, 30)            # US session open in New York time (= 13:30 UTC in summer, 14:30 UTC in winter)
OPENING_RANGE_MINUTES = 30      # opening range = first 30 minutes after the US open
# Direction AUC (validation) of the run with RSI 7/14/21/50 and TWAP/VWAP 15/60 (no session structure) – reference in 6.2.
# New periods are only worth keeping if the validation AUC rises by at least MIN_AUC_GAIN.
REFERENCE_DIRECTION_AUC = {"15min": 0.5228, "60min": 0.5152}
MIN_AUC_GAIN = 0.005

# Trade setups (horizon in minutes). Two kinds of barriers:
#   mode "vol":   target/stop = sigma x expected move over the horizon
#                 (std of 1-minute close changes over the last VOL_LOOKBACK candles x sqrt(horizon)), in ticks
#                 -> a trade means the same in quiet and in wild markets
#   mode "fixed": target_ticks / stop_ticks as fixed numbers, e.g.
#                 {"name": "15min_fix", "horizon": 15, "mode": "fixed", "target_ticks": 30, "stop_ticks": 30}
SETUPS = [
    {"name": "15min", "horizon": 15, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
    {"name": "60min", "horizon": 60, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
]
VOL_LOOKBACK = 60               # candles for the volatility estimate (only past data)
MIN_BARRIER_TICKS = 5           # volatility barriers are never smaller than this

# Futures contract and execution
TICK_SIZE = 0.10                # GC / MGC
TICK_VALUE_USD = 10.0           # GC: 10 USD per tick and contract (MGC: 1 USD)
COST_PER_TRADE_TICKS = 2.0      # round trip incl. spread/slippage and commission -> adjust to your account
TARGET_FILL_EXTRA_TICKS = 1     # target counts only if the price trades this many ticks beyond it
MAX_GAP_MINUTES = 5             # no trade if the window contains a longer pause (session break, weekend)
WARMUP_CANDLES = 240            # first candles of a file are not traded (longest rolling input: 240 minutes)

# Data split and training
VALIDATION_FRACTION = 0.2       # last 20 % of the training data (before the test period) -> threshold selection
TRAIN_ROW_STRIDE = 1            # 1 = every candle; e.g. 3 = every 3rd candle (faster, slightly less data)
GBM_PARAMS = dict(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=500,
    l2_regularization=1.0, random_state=42,
)

# Trading rule: trade if max(P_long, P_short) >= threshold; the threshold is chosen on validation (best t-value)
PROB_THRESHOLDS = [round(0.30 + 0.025 * i, 3) for i in range(21)]   # 0.30 ... 0.80
MIN_VALIDATION_TRADES = 100     # thresholds with fewer validation trades are not eligible
BASELINE_RANDOM_RUNS = 5
COMPUTE_FEATURE_IMPORTANCE = True   # permutation importance on a validation sample (a few minutes)

# Direction strategy (section 8.2): direction from the direction model, timing from the win models.
# All combinations are evaluated on validation; the one with the best t-value is applied ONCE to the test data.
DIRECTION_QUANTILES = [0.05, 0.10, 0.20]   # trade the lowest / highest X % of P(up first) -> short / long
TIMING_QUANTILES = [0.0, 0.5, 0.75]        # also require P(win) of the traded side >= this validation quantile (0 = off)
DIRECTION_SIDES = ["both", "short", "long"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
for p in (TRAIN_CSV_PATH, TEST_CSV_PATH):
    if not p.is_file():
        raise FileNotFoundError(p)
print(f"Train CSV: {TRAIN_CSV_PATH}\nTest CSV:  {TEST_CSV_PATH}")
for s in SETUPS:
    if s.get("mode", "fixed") == "vol":
        print(f"Setup {s['name']}: target {s['target_sigma']} σ / stop {s['stop_sigma']} σ of the expected "
              f"{s['horizon']}-minute move (ticks are shown after the labels are computed)")
    else:
        be = (s["stop_ticks"] + COST_PER_TRADE_TICKS) / (s["target_ticks"] + s["stop_ticks"])
        print(f"Setup {s['name']}: +{s['target_ticks']} / -{s['stop_ticks']} ticks within {s['horizon']} min "
              f"-> break-even win rate ≈ {100 * be:.1f} % (ignoring time exits)")

Train CSV: E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv
Test CSV:  E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv
Setup 15min: target 1.0 σ / stop 1.0 σ of the expected 15-minute move (ticks are shown after the labels are computed)
Setup 60min: target 1.0 σ / stop 1.0 σ of the expected 60-minute move (ticks are shown after the labels are computed)


## 1. Imports

In [2]:
import json
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except ImportError:  # outside Jupyter
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

## 2. Load CSV

In [3]:
def load_csv(csv_path: Path) -> pd.DataFrame:
    """Load an ATAS 1-minute export in Bar order; only the input columns, the timestamp and Bar are kept."""
    raw = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
    missing = [c for c in ["Bar", "Time UTC"] + INPUT_COLUMNS if c not in raw.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")

    for col in PERCENT_COLUMNS:
        text = (raw[col].astype(str).str.strip().str.replace("%", "", regex=False)
                .str.replace("'", "", regex=False).str.replace(" ", "", regex=False))
        text = text.where(text.str.contains(".", regex=False), text.str.replace(",", ".", regex=False))
        values = pd.to_numeric(text, errors="coerce")
        n_bad = int((values.isna() & (text != "") & (text.str.lower() != "nan")).sum())
        if n_bad:
            print(f"⚠ {col}: {n_bad:,} values not parseable -> 0")
        raw[col] = (values / 100).fillna(0.0)

    df = raw[["Bar", "Time UTC"] + INPUT_COLUMNS].copy()
    for col in INPUT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    df["timestamp"] = pd.to_datetime(df["Time UTC"], format="%d.%m.%Y %H:%M:%S", utc=True)
    if not df["Bar"].is_monotonic_increasing:
        df = df.sort_values("Bar", kind="stable")
    df = df.drop(columns=["Time UTC"]).reset_index(drop=True)
    if df[["Open", "High", "Low", "Close"]].isna().any().any():
        raise ValueError(f"{csv_path.name}: missing OHLC values")
    print(f"{csv_path.name}: {len(df):,} candles, {df['timestamp'].min()} → {df['timestamp'].max()}")
    return df


df_all = load_csv(TRAIN_CSV_PATH)
df_test = load_csv(TEST_CSV_PATH)

# Training data = everything before the first candle of the test CSV (no overlap)
test_start = df_test["timestamp"].min()
df_train = df_all[df_all["timestamp"] < test_start].reset_index(drop=True)
del df_all
print(f"Training data used: {len(df_train):,} candles until {df_train['timestamp'].max()} (test starts {test_start})")

Candelexport_GC_M1_25_11_2018_20_10_2026.csv: 2,796,096 candles, 2018-11-25 23:00:00+00:00 → 2026-10-20 20:59:00+00:00
Test_GC_M1_20_10_2025_20_10_2026.csv: 354,649 candles, 2025-10-20 00:00:00+00:00 → 2026-10-20 20:59:00+00:00
Training data used: 2,441,447 candles until 2025-10-19 23:59:00+00:00 (test starts 2025-10-20 00:00:00+00:00)


## 3. Inputs (features)

Only backward-looking values of the input columns: candle shape, price changes, volatility and position in the recent range
(all in ticks, independent of the price level), plus order flow (Ask/Bid, Delta, Session Delta, Max/Min Delta, Volume)
as raw values, ratios and rolling sums. Gradient boosting handles missing values itself.

**Indicators:** RSI for every period in `RSI_PERIODS` (level + change over 5 minutes), **TWAP** and **VWAP** – each as
session value (reset at every session start / after a pause) and as rolling value over `TWAP_WINDOWS` / `VWAP_WINDOWS`.
TWAP/VWAP enter as **distance of the close in ticks** (and relative to the recent candle range), so the model does not
see absolute price levels. Typical price = (High + Low + Close) / 3.

**Session structure** (trading day = 18:00 → 17:00 New York time, like CME Globex):
* **Previous session:** high, low, close and VWAP → distance of the current close in ticks, position inside the previous
  range, previous range; Session Delta at the end of the previous session, its delta sum and delta per volume.
* **Night session (Asia/London):** from the session start until the US open – range so far, position inside it, distance
  to night high/low; after the US open the final night range is used. Flag whether the candle is in the night session.
* **US opening range:** high/low of the first `OPENING_RANGE_MINUTES` after the US open; available only **after** it is
  complete (no look-ahead): range, distance to OR high/low, +1 above / −1 below / 0 inside. Minutes since the US open.

All values use only completed data up to the current candle.

In [4]:
def rsi(close: pd.Series, period: int) -> pd.Series:
    """Relative Strength Index with Wilder smoothing (0..100)."""
    diff = close.diff()
    avg_gain = diff.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = (-diff.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    value = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))
    value[(avg_loss == 0) & (avg_gain > 0)] = 100.0
    value[(avg_loss == 0) & (avg_gain == 0)] = 50.0
    return value


def session_structure_features(df: pd.DataFrame) -> dict:
    """Previous session, night session and US opening range (New York time, DST-aware)."""
    t = TICK_SIZE
    c, h, l, vol = df["Close"], df["High"], df["Low"], df["Volume"]
    ts_et = df["timestamp"].dt.tz_convert("America/New_York")
    trade_day = (ts_et + pd.Timedelta(hours=24 - SESSION_START_HOUR_ET)).dt.date   # 18:00 ET -> next trading day
    tod = ts_et.dt.hour * 60 + ts_et.dt.minute
    us_open = US_OPEN_ET[0] * 60 + US_OPEN_ET[1]
    same_calendar_day = ts_et.dt.date == trade_day
    f = {}

    # ---- previous session (completed trading day before the current one)
    tp = (h + l + c) / 3
    g = pd.DataFrame({"day": trade_day, "h": h, "l": l, "c": c, "pv": tp * vol, "v": vol,
                      "sd": df["Session Delta"], "d": df["Delta"]})
    daily = g.groupby("day", sort=True).agg(high=("h", "max"), low=("l", "min"), close=("c", "last"),
                                            pv=("pv", "sum"), v=("v", "sum"), sd_end=("sd", "last"),
                                            delta_sum=("d", "sum"))
    daily["vwap"] = daily["pv"] / daily["v"].replace(0, np.nan)
    prev = daily.shift(1).reindex(trade_day.to_numpy())
    prev.index = df.index
    prev_range = (prev["high"] - prev["low"]).replace(0, np.nan)
    f["dist_prev_high_ticks"] = (c - prev["high"]) / t
    f["dist_prev_low_ticks"] = (c - prev["low"]) / t
    f["dist_prev_close_ticks"] = (c - prev["close"]) / t
    f["dist_prev_vwap_ticks"] = (c - prev["vwap"]) / t
    f["prev_range_ticks"] = prev_range / t
    f["pos_in_prev_range"] = (c - prev["low"]) / prev_range
    f["prev_session_delta"] = prev["sd_end"]
    f["prev_delta_sum"] = prev["delta_sum"]
    f["prev_delta_per_volume"] = prev["delta_sum"] / prev["v"].replace(0, np.nan)

    # ---- night session: session start until the US open (running high/low, fixed after the open)
    is_night = (~same_calendar_day) | (tod < us_open)
    night_high = h.where(is_night).groupby(trade_day).cummax().groupby(trade_day).ffill()
    night_low = l.where(is_night).groupby(trade_day).cummin().groupby(trade_day).ffill()
    night_range = (night_high - night_low).replace(0, np.nan)
    f["is_night_session"] = is_night.astype("float32")
    f["night_range_ticks"] = night_range / t
    f["pos_in_night_range"] = (c - night_low) / night_range
    f["dist_night_high_ticks"] = (c - night_high) / t
    f["dist_night_low_ticks"] = (c - night_low) / t

    # ---- US opening range: only after it is complete
    in_or = same_calendar_day & (tod >= us_open) & (tod < us_open + OPENING_RANGE_MINUTES)
    or_high = h.where(in_or).groupby(trade_day).transform("max")
    or_low = l.where(in_or).groupby(trade_day).transform("min")
    or_done = same_calendar_day & (tod >= us_open + OPENING_RANGE_MINUTES)
    or_high, or_low = or_high.where(or_done), or_low.where(or_done)
    f["or_range_ticks"] = (or_high - or_low) / t
    f["dist_or_high_ticks"] = (c - or_high) / t
    f["dist_or_low_ticks"] = (c - or_low) / t
    f["or_position"] = np.where(or_high.isna(), np.nan, np.where(c > or_high, 1.0, np.where(c < or_low, -1.0, 0.0)))
    f["minutes_since_us_open"] = np.where(same_calendar_day, tod - us_open, tod - us_open - 1440).astype("float32")
    return f


def session_ids(ts: pd.Series) -> pd.Series:
    """New session after every pause > MAX_GAP_MINUTES (daily break, weekend)."""
    return (ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES).cumsum()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    vol, ask, bid, delta = df["Volume"], df["Ask"], df["Bid"], df["Delta"]
    t = TICK_SIZE
    f = {}

    # Candle shape and price changes (ticks)
    f["body_ticks"] = (c - o) / t
    f["range_ticks"] = (h - l) / t
    f["upper_wick_ticks"] = (h - np.maximum(o, c)) / t
    f["lower_wick_ticks"] = (np.minimum(o, c) - l) / t
    ret1 = (c - c.shift(1)) / t
    for k in [1, 3, 5, 10, 15, 30, 60]:
        f[f"ret_{k}_ticks"] = (c - c.shift(k)) / t
    for k in [5, 15, 60]:
        hh, ll = h.rolling(k).max(), l.rolling(k).min()
        f[f"range_mean_{k}"] = f["range_ticks"].rolling(k).mean()
        f[f"ret_std_{k}"] = ret1.rolling(k).std()
        f[f"pos_in_range_{k}"] = (c - ll) / (hh - ll).replace(0, np.nan)
        f[f"dist_high_{k}_ticks"] = (hh - c) / t
        f[f"dist_low_{k}_ticks"] = (c - ll) / t

    # Order flow
    f["volume"] = vol
    f["log_volume"] = np.log1p(vol.clip(lower=0))
    f["ask"] = ask
    f["bid"] = bid
    f["ask_bid_imbalance"] = (ask - bid) / (ask + bid).replace(0, np.nan)
    f["delta"] = delta
    f["delta_per_volume"] = df["Delta/Volume"]
    f["session_delta"] = df["Session Delta"]
    f["session_delta_pct"] = df["Session Delta %"]
    f["session_delta_change_15"] = df["Session Delta"] - df["Session Delta"].shift(15)
    f["max_delta"] = df["Max Delta"]
    f["min_delta"] = df["Min Delta"]
    f["delta_change"] = df["Delta Change"]
    f["max_delta_rel"] = df["Max Delta"] / vol.replace(0, np.nan)
    f["min_delta_rel"] = df["Min Delta"] / vol.replace(0, np.nan)
    for k in [1, 2, 3, 5]:
        f[f"delta_lag_{k}"] = delta.shift(k)
        f[f"delta_per_volume_lag_{k}"] = df["Delta/Volume"].shift(k)
    for k in [5, 15, 60]:
        f[f"delta_sum_{k}"] = delta.rolling(k).sum()
        f[f"delta_ratio_{k}"] = delta.rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
        f[f"volume_rel_{k}"] = vol / vol.rolling(k).mean().replace(0, np.nan)
        f[f"ask_bid_imbalance_{k}"] = ((ask - bid).rolling(k).sum()
                                       / (ask + bid).rolling(k).sum().replace(0, np.nan))

    # RSI
    for p in RSI_PERIODS:
        r = rsi(c, p)
        f[f"rsi_{p}"] = r
        f[f"rsi_{p}_change_5"] = r - r.shift(5)

    # TWAP / VWAP (distance of the close in ticks and relative to the recent candle range)
    tp = (h + l + c) / 3
    session = session_ids(df["timestamp"])
    range_60 = ((h - l) / t).rolling(60, min_periods=5).mean().replace(0, np.nan)
    twap_session = tp.groupby(session).cumsum() / (tp.groupby(session).cumcount() + 1)
    vol_cum = vol.groupby(session).cumsum()
    vwap_session = ((tp * vol).groupby(session).cumsum() / vol_cum.replace(0, np.nan)).groupby(session).ffill()
    averages = {"twap_session": twap_session, "vwap_session": vwap_session}
    for k in TWAP_WINDOWS:
        averages[f"twap_{k}"] = tp.rolling(k).mean()
    for k in VWAP_WINDOWS:
        averages[f"vwap_{k}"] = (tp * vol).rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
    for name, avg in averages.items():
        f[f"dist_{name}_ticks"] = (c - avg) / t
        f[f"dist_{name}_rel"] = (c - avg) / t / range_60
    f["vwap_minus_twap_session_ticks"] = (vwap_session - twap_session) / t

    # Session structure over several days
    f.update(session_structure_features(df))

    # Time of day / weekday from the timestamp (UTC)
    if USE_TIME_FEATURES:
        ts = df["timestamp"]
        minute_of_day = ts.dt.hour * 60 + ts.dt.minute
        f["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
        f["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
        f["weekday"] = ts.dt.dayofweek
        f["minutes_since_gap"] = minutes_since_gap(ts)

    out = pd.DataFrame(f).replace([np.inf, -np.inf], np.nan).astype("float32")
    return out


def minutes_since_gap(ts: pd.Series) -> pd.Series:
    """Minutes since the last pause > MAX_GAP_MINUTES (session start / after the weekend)."""
    gap = ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES
    block = gap.cumsum()
    return ts.groupby(block).cumcount().astype("float32")


t0 = time.time()
X_train_all = build_features(df_train)
X_test = build_features(df_test)
FEATURE_COLUMNS = list(X_train_all.columns)
print(f"{len(FEATURE_COLUMNS)} inputs built in {time.time() - t0:.0f} s")
SESSION_PREFIXES = ("rsi_", "dist_twap", "dist_vwap", "vwap_minus", "dist_prev", "prev_", "pos_in_prev", "is_night",
                    "night_", "pos_in_night", "dist_night", "or_", "dist_or", "minutes_since_us_open")
print("Indicator and session inputs:", [c for c in FEATURE_COLUMNS if c.startswith(SESSION_PREFIXES)])

103 inputs built in 7 s
Indicator and session inputs: ['rsi_100', 'rsi_100_change_5', 'dist_twap_session_ticks', 'dist_twap_session_rel', 'dist_vwap_session_ticks', 'dist_vwap_session_rel', 'dist_twap_60_ticks', 'dist_twap_60_rel', 'dist_twap_120_ticks', 'dist_twap_120_rel', 'dist_twap_240_ticks', 'dist_twap_240_rel', 'dist_vwap_60_ticks', 'dist_vwap_60_rel', 'dist_vwap_120_ticks', 'dist_vwap_120_rel', 'dist_vwap_240_ticks', 'dist_vwap_240_rel', 'vwap_minus_twap_session_ticks', 'dist_prev_high_ticks', 'dist_prev_low_ticks', 'dist_prev_close_ticks', 'dist_prev_vwap_ticks', 'prev_range_ticks', 'pos_in_prev_range', 'prev_session_delta', 'prev_delta_sum', 'prev_delta_per_volume', 'is_night_session', 'night_range_ticks', 'pos_in_night_range', 'dist_night_high_ticks', 'dist_night_low_ticks', 'or_range_ticks', 'dist_or_high_ticks', 'dist_or_low_ticks', 'or_position', 'minutes_since_us_open']


## 4. Labels: Barrier Outcomes for Every Candle

For every candle, a long and a short trade entered at its close are played through minute by minute:
target (+`TARGET_FILL_EXTRA_TICKS` beyond it) → win, stop → loss (stop wins if both are touched in the same candle),
neither → closed at the close after `horizon` minutes. Windows that contain a pause are invalid (not traded, not trained).

With `mode: "vol"` target and stop are set per candle from the volatility of the **past** `VOL_LOOKBACK` candles.

**Direction label (for the pure direction test):** is the upper barrier (+target) touched **before** the lower one
(−target)? Only windows where exactly one side is reached first count; time-outs and candles touching both are excluded.
This removes the volatility question – what remains is only "up or down".

In [5]:
EXIT_TARGET, EXIT_STOP, EXIT_TIME = 1, 2, 3


def barrier_ticks(df: pd.DataFrame, setup: dict):
    """Target and stop in ticks per candle (arrays)."""
    n = len(df)
    if setup.get("mode", "fixed") == "fixed":
        return np.full(n, float(setup["target_ticks"])), np.full(n, float(setup["stop_ticks"]))
    ret1 = df["Close"].diff() / TICK_SIZE
    expected_move = (ret1.rolling(VOL_LOOKBACK, min_periods=VOL_LOOKBACK // 2).std() * np.sqrt(setup["horizon"])).to_numpy()
    target = np.maximum(np.round(setup["target_sigma"] * expected_move), MIN_BARRIER_TICKS)
    stop = np.maximum(np.round(setup["stop_sigma"] * expected_move), MIN_BARRIER_TICKS)
    return target, stop


def barrier_outcomes(df: pd.DataFrame, setup: dict) -> dict:
    horizon = setup["horizon"]
    target_ticks, stop_ticks = barrier_ticks(df, setup)
    c = df["Close"].to_numpy("float64")
    h = df["High"].to_numpy("float64")
    l = df["Low"].to_numpy("float64")
    n = len(df)
    extra = TARGET_FILL_EXTRA_TICKS * TICK_SIZE
    out = {"target_ticks": target_ticks, "stop_ticks": stop_ticks}
    for side, d in (("long", 1), ("short", -1)):
        target = c + d * (target_ticks * TICK_SIZE + extra)
        stop = c - d * stop_ticks * TICK_SIZE
        result = np.full(n, np.nan)
        exit_type = np.zeros(n, dtype="int8")
        minutes = np.zeros(n, dtype="int16")
        done = np.zeros(n, dtype=bool)
        for k in range(1, horizon + 1):
            hk = np.full(n, np.nan)
            lk = np.full(n, np.nan)
            hk[:n - k], lk[:n - k] = h[k:], l[k:]
            if d == 1:
                stop_hit, tgt_hit = lk <= stop, hk >= target
            else:
                stop_hit, tgt_hit = hk >= stop, lk <= target
            stop_hit &= ~done
            tgt_hit &= ~done & ~stop_hit
            result[stop_hit], exit_type[stop_hit], minutes[stop_hit] = -stop_ticks[stop_hit], EXIT_STOP, k
            result[tgt_hit], exit_type[tgt_hit], minutes[tgt_hit] = target_ticks[tgt_hit], EXIT_TARGET, k
            done |= stop_hit | tgt_hit
        ck = np.full(n, np.nan)
        ck[:n - horizon] = c[horizon:]
        open_ = ~done
        result[open_] = d * (ck[open_] - c[open_]) / TICK_SIZE
        exit_type[open_], minutes[open_] = EXIT_TIME, horizon
        out[side] = {"result": result, "exit": exit_type, "minutes": minutes, "win": exit_type == EXIT_TARGET}

    # Direction label: +target before -target (symmetric, no extra tick); both in one candle / time-out -> unresolved
    up_level, down_level = c + target_ticks * TICK_SIZE, c - target_ticks * TICK_SIZE
    up_first = np.zeros(n, dtype=bool)
    resolved = np.zeros(n, dtype=bool)
    decided = np.zeros(n, dtype=bool)
    for k in range(1, horizon + 1):
        hk = np.full(n, np.nan)
        lk = np.full(n, np.nan)
        hk[:n - k], lk[:n - k] = h[k:], l[k:]
        up, down = (hk >= up_level) & ~decided, (lk <= down_level) & ~decided
        only_up, only_down = up & ~down, down & ~up
        up_first |= only_up
        resolved |= only_up | only_down
        decided |= up | down
    out["direction"] = {"up_first": up_first, "resolved": resolved}

    # Valid = complete window inside the file, no pause inside the window, warm-up done
    gap = (df["timestamp"].diff().dt.total_seconds().div(60).fillna(0).to_numpy() > MAX_GAP_MINUTES).astype(int)
    cum = np.cumsum(gap)
    idx = np.arange(n)
    end = np.minimum(idx + horizon, n - 1)
    out["valid"] = ((idx + horizon < n) & ((cum[end] - cum[idx]) == 0) & (idx >= WARMUP_CANDLES)
                    & np.isfinite(target_ticks) & np.isfinite(stop_ticks))
    return out


t0 = time.time()
outcomes_train = {s["name"]: barrier_outcomes(df_train, s) for s in SETUPS}
outcomes_test = {s["name"]: barrier_outcomes(df_test, s) for s in SETUPS}
print(f"Barrier outcomes computed in {time.time() - t0:.0f} s")
for s in SETUPS:
    o = outcomes_train[s["name"]]
    v = o["valid"]
    print(f"[{s['name']}] valid candles {v.sum():,} | win rate if ALWAYS traded: "
          f"long {100 * o['long']['win'][v].mean():.1f} %, short {100 * o['short']['win'][v].mean():.1f} % "
          f"| Ø result long {np.nanmean(o['long']['result'][v]):+.2f}, short {np.nanmean(o['short']['result'][v]):+.2f} ticks (gross)")
    for label, oo in (("training data", o), ("test data", outcomes_test[s["name"]])):
        vv = oo["valid"]
        print(f"      {label}: Ø target {np.mean(oo['target_ticks'][vv]):.1f} / Ø stop {np.mean(oo['stop_ticks'][vv]):.1f} ticks, "
              f"direction resolved in {100 * oo['direction']['resolved'][vv].mean():.1f} % of the windows, "
              f"up first {100 * oo['direction']['up_first'][vv & oo['direction']['resolved']].mean():.1f} %")

Barrier outcomes computed in 4 s
in] valid candles 2,414,279 | win rate if ALWAYS traded: long 24.6 %, short 25.1 % | Ø result long -0.25, short -0.29 ticks (gross)
      training data: Ø target 18.0 / Ø stop 18.0 ticks, direction resolved in 56.2 % of the windows, up first 49.6 %
      test data: Ø target 71.6 / Ø stop 71.6 ticks, direction resolved in 55.5 % of the windows, up first 47.8 %
in] valid candles 2,333,548 | win rate if ALWAYS traded: long 26.6 %, short 27.4 % | Ø result long -0.21, short -0.37 ticks (gross)
      training data: Ø target 36.2 / Ø stop 36.2 ticks, direction resolved in 57.2 % of the windows, up first 49.3 %
      test data: Ø target 144.3 / Ø stop 144.3 ticks, direction resolved in 56.8 % of the windows, up first 46.8 %


## 5. Time Split: Training / Validation

In [6]:
n_train_total = len(df_train)
val_start = int(n_train_total * (1 - VALIDATION_FRACTION))
max_horizon = max(s["horizon"] for s in SETUPS)
# Purge: training labels must not look into the validation period
train_rows = np.arange(0, val_start - max_horizon, TRAIN_ROW_STRIDE)
val_rows = np.arange(val_start, n_train_total)
print(f"Training:   {df_train['timestamp'].iloc[train_rows[0]]} → {df_train['timestamp'].iloc[train_rows[-1]]} "
      f"({len(train_rows):,} candles, stride {TRAIN_ROW_STRIDE})")
print(f"Validation: {df_train['timestamp'].iloc[val_rows[0]]} → {df_train['timestamp'].iloc[val_rows[-1]]} ({len(val_rows):,} candles)")
print(f"Test:       {df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]} ({len(df_test):,} candles)")

Training:   2018-11-25 23:00:00+00:00 → 2024-06-04 04:27:00+00:00 (1,953,097 candles, stride 1)
Validation: 2024-06-04 05:29:00+00:00 → 2025-10-19 23:59:00+00:00 (488,290 candles)
Test:       2025-10-20 00:00:00+00:00 → 2026-10-20 20:59:00+00:00 (354,649 candles)


## 6. Train Long and Short Classifier per Setup

In [7]:
models = {}
auc_rows = []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    tr = train_rows[o_tr["valid"][train_rows]]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    models[name] = {}
    for side in ("long", "short"):
        t0 = time.time()
        clf = HistGradientBoostingClassifier(**GBM_PARAMS)
        clf.fit(X_train_all.iloc[tr], o_tr[side]["win"][tr])
        p_val = clf.predict_proba(X_train_all.iloc[va])[:, 1]
        p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
        models[name][side] = clf
        auc_rows.append({
            "Setup": name, "Side": side, "Train candles": len(tr),
            "Base win rate val %": 100 * o_tr[side]["win"][va].mean(),
            "AUC validation": roc_auc_score(o_tr[side]["win"][va], p_val),
            "AUC test": roc_auc_score(o_te[side]["win"][te], p_te),
            "Ø P val": p_val.mean(), "P val 99%": np.percentile(p_val, 99),
            "Fit seconds": time.time() - t0,
        })
        print(f"[{name} {side}] trained in {time.time() - t0:.0f} s – AUC validation {auc_rows[-1]['AUC validation']:.4f}, "
              f"test {auc_rows[-1]['AUC test']:.4f}")

print("\nAUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful")
display(pd.DataFrame(auc_rows).round(4))

in long] trained in 33 s – AUC validation 0.6020, test 0.5899
in short] trained in 31 s – AUC validation 0.6109, test 0.5820
in long] trained in 27 s – AUC validation 0.6108, test 0.5839
in short] trained in 27 s – AUC validation 0.6128, test 0.5720

AUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful


,Setup,Side,Train candles,Base win rate val %,AUC validation,AUC test,Ø P val,P val 99%,Fit seconds
0,15min,long,1931344,26.7251,0.6020,0.5899,0.2444,0.4329,32.9773
1,15min,short,1931344,26.9117,0.6109,0.5820,0.2579,0.4449,31.4279
2,60min,long,1866858,28.1934,0.6108,0.5839,0.2445,0.4853,27.4835
3,60min,short,1866858,27.9958,0.6128,0.5720,0.2728,0.5313,26.8119


### 6.1 Which Inputs Matter (permutation importance on a validation sample)

In [8]:
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(0)
    for s in SETUPS:
        name = s["name"]
        va = val_rows[outcomes_train[name]["valid"][val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        for side in ("long", "short"):
            imp = permutation_importance(
                models[name][side], X_train_all.iloc[sample], outcomes_train[name][side]["win"][sample],
                scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
            )
            top = (pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False).head(12))
            print(f"[{name} {side}] AUC loss when an input is shuffled (top 12):")
            print("   " + ", ".join(f"{k} {v:.4f}" for k, v in top.items()))

in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0769, volume_rel_60 0.0132, tod_cos 0.0128, minutes_since_us_open 0.0117, tod_sin 0.0097, range_mean_15 0.0082, range_mean_60 0.0076, minutes_since_gap 0.0048, volume 0.0016, vwap_minus_twap_session_ticks 0.0010, dist_or_low_ticks 0.0009, range_mean_5 0.0008
in short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0719, range_mean_15 0.0162, minutes_since_us_open 0.0147, volume_rel_60 0.0116, range_mean_60 0.0109, tod_cos 0.0109, tod_sin 0.0063, minutes_since_gap 0.0030, range_mean_5 0.0023, dist_low_60_ticks 0.0020, prev_range_ticks 0.0019, dist_or_low_ticks 0.0017
in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0843, tod_cos 0.0194, minutes_since_us_open 0.0134, minutes_since_gap 0.0051, volume_rel_60 0.0040, range_mean_15 0.0036, tod_sin 0.0036, vwap_minus_twap_session_ticks 0.0035, dist_or_high_ticks 0.0023, dist_or_low_ticks 0.0021, prev_range_ticks 0.0017, prev_session_delta 

### 6.2 Pure Direction Test: "Up or Down First?"

One classifier per setup on the direction label only (volatility removed). **AUC 0.50 = no direction information**,
0.52–0.55 = weak, above 0.55 = interesting. The bucket table shows on the test data whether higher model probabilities
really go along with more "up first" outcomes – that is what a trading signal needs.

In [9]:
direction_models, direction_rows = {}, []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    ok_tr = o_tr["valid"] & o_tr["direction"]["resolved"]
    ok_te = o_te["valid"] & o_te["direction"]["resolved"]
    tr = train_rows[ok_tr[train_rows]]
    va = val_rows[ok_tr[val_rows]]
    te = np.flatnonzero(ok_te)
    t0 = time.time()
    clf = HistGradientBoostingClassifier(**GBM_PARAMS)
    clf.fit(X_train_all.iloc[tr], o_tr["direction"]["up_first"][tr])
    direction_models[name] = clf
    p_va = clf.predict_proba(X_train_all.iloc[va])[:, 1]
    p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
    y_va, y_te = o_tr["direction"]["up_first"][va], o_te["direction"]["up_first"][te]
    direction_rows.append({
        "Setup": name, "Train windows": len(tr), "Up first val %": 100 * y_va.mean(), "Up first test %": 100 * y_te.mean(),
        "AUC direction validation": roc_auc_score(y_va, p_va), "AUC direction test": roc_auc_score(y_te, p_te),
        "Fit seconds": time.time() - t0,
    })
    buckets = pd.qcut(p_te, 10, labels=False, duplicates="drop")
    table = (pd.DataFrame({"bucket": buckets, "p": p_te, "up": y_te})
             .groupby("bucket").agg(windows=("up", "size"), mean_p=("p", "mean"), actual_up_pct=("up", "mean")))
    table["actual_up_pct"] *= 100
    print(f"[{name}] test: model probability (10 buckets, low -> high) vs. actual share 'up first'")
    display(table.round(3))

print("AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting")
direction_df = pd.DataFrame(direction_rows)
display(direction_df.round(4))

# Comparison with the previous run (validation only – the test data is not used for this decision)
for _, r in direction_df.iterrows():
    ref = REFERENCE_DIRECTION_AUC.get(r["Setup"])
    if ref is None:
        continue
    gain = r["AUC direction validation"] - ref
    verdict = ("new inputs HELP – direction information improved" if gain >= MIN_AUC_GAIN
               else "no relevant gain over the reference run")
    print(f"[{r['Setup']}] direction AUC validation {r['AUC direction validation']:.4f} vs. reference {ref:.4f} "
          f"-> {gain:+.4f} → {verdict}")

# Which inputs does the DIRECTION model use? (permutation importance on resolved validation windows)
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(1)
    indicator_prefixes = SESSION_PREFIXES
    for s in SETUPS:
        name = s["name"]
        o_tr = outcomes_train[name]
        ok = o_tr["valid"] & o_tr["direction"]["resolved"]
        va = val_rows[ok[val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        imp = permutation_importance(
            direction_models[name], X_train_all.iloc[sample], o_tr["direction"]["up_first"][sample],
            scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
        )
        importance = pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False)
        print(f"\n[{name} DIRECTION] top 15 inputs (AUC loss when shuffled):")
        print("   " + ", ".join(f"{k} {v:.4f}" for k, v in importance.head(15).items()))
        ind = importance[[c for c in importance.index if c.startswith(indicator_prefixes)]]
        print(f"[{name} DIRECTION] all indicator and session inputs, best first:")
        display(ind.round(5).to_frame("AUC loss").T)

in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19442,0.362,45.129
1,19441,0.408,46.685
2,19442,0.431,47.727
3,19441,0.450,47.420
4,19442,0.465,47.732
5,19441,0.479,47.714
6,19441,0.494,48.768
7,19442,0.510,48.673
8,19441,0.531,49.231


in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19242,0.217,44.283
1,19242,0.292,46.383
2,19240,0.342,47.323
3,19242,0.384,46.097
4,19241,0.418,46.292
5,19241,0.447,47.789
6,19242,0.476,48.467
7,19241,0.507,46.905
8,19241,0.550,46.328


AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting


,Setup,Train windows,Up first val %,Up first test %,AUC direction validation,AUC direction test,Fit seconds
0,15min,1080353,49.9036,47.8425,0.5160,0.5131,13.7346
1,60min,1064210,50.1908,46.7773,0.5165,0.5075,13.5490


in] direction AUC validation 0.5160 vs. reference 0.5228 -> -0.0068 → no relevant gain over the reference run
in] direction AUC validation 0.5165 vs. reference 0.5152 -> +0.0013 → no relevant gain over the reference run

in DIRECTION] top 15 inputs (AUC loss when shuffled):
   or_range_ticks 0.0037, tod_sin 0.0035, dist_night_high_ticks 0.0033, minutes_since_gap 0.0030, vwap_minus_twap_session_ticks 0.0027, pos_in_prev_range 0.0026, dist_vwap_120_rel 0.0025, dist_high_60_ticks 0.0021, dist_prev_low_ticks 0.0018, prev_delta_per_volume 0.0015, range_mean_60 0.0015, night_range_ticks 0.0013, ret_std_60 0.0013, ret_std_15 0.0012, tod_cos 0.0012
in DIRECTION] all indicator and session inputs, best first:


,or_range_ticks,dist_night_high_ticks,vwap_minus_twap_session_ticks,pos_in_prev_range,dist_vwap_120_rel,dist_prev_low_ticks,prev_delta_per_volume,night_range_ticks,dist_twap_session_ticks,dist_night_low_ticks,dist_twap_120_rel,prev_range_ticks,dist_or_low_ticks,dist_vwap_60_ticks,rsi_100,dist_twap_120_ticks,dist_vwap_session_ticks,minutes_since_us_open,dist_or_high_ticks,dist_twap_60_ticks,dist_twap_60_rel,dist_twap_session_rel,is_night_session,prev_delta_sum,or_position,rsi_100_change_5,pos_in_night_range,dist_vwap_60_rel,dist_vwap_session_rel,dist_vwap_240_rel,dist_vwap_120_ticks,dist_vwap_240_ticks,dist_twap_240_ticks,dist_prev_high_ticks,dist_twap_240_rel,dist_prev_vwap_ticks,dist_prev_close_ticks,prev_session_delta
AUC loss,0.00369,0.00334,0.0027,0.00263,0.00249,0.00183,0.00154,0.0013,0.00096,0.00094,0.00093,0.00075,0.00063,0.00042,0.00041,0.00032,0.00031,0.0003,0.00027,0.00017,0.00015,0.0,0.0,0.0,0.0,-0.00002,-0.00004,-0.00006,-0.00007,-0.00014,-0.00017,-0.00022,-0.00082,-0.00087,-0.00093,-0.00119,-0.00126,-0.00158



in DIRECTION] top 15 inputs (AUC loss when shuffled):
   prev_delta_per_volume 0.0052, dist_high_60_ticks 0.0043, vwap_minus_twap_session_ticks 0.0042, prev_range_ticks 0.0039, ret_std_60 0.0029, range_mean_60 0.0027, dist_prev_high_ticks 0.0023, prev_session_delta 0.0020, pos_in_prev_range 0.0020, dist_prev_low_ticks 0.0018, tod_sin 0.0016, dist_prev_vwap_ticks 0.0016, dist_vwap_60_ticks 0.0016, dist_vwap_120_ticks 0.0013, dist_vwap_session_ticks 0.0012
in DIRECTION] all indicator and session inputs, best first:


,prev_delta_per_volume,vwap_minus_twap_session_ticks,prev_range_ticks,dist_prev_high_ticks,prev_session_delta,pos_in_prev_range,dist_prev_low_ticks,dist_prev_vwap_ticks,dist_vwap_60_ticks,dist_vwap_120_ticks,dist_vwap_session_ticks,dist_night_low_ticks,dist_twap_240_rel,dist_twap_session_rel,or_range_ticks,dist_vwap_session_rel,dist_or_high_ticks,night_range_ticks,dist_twap_60_ticks,dist_vwap_240_ticks,dist_twap_60_rel,dist_vwap_60_rel,dist_vwap_240_rel,dist_or_low_ticks,rsi_100_change_5,is_night_session,or_position,prev_delta_sum,pos_in_night_range,dist_twap_120_ticks,dist_twap_120_rel,minutes_since_us_open,dist_twap_240_ticks,dist_twap_session_ticks,dist_vwap_120_rel,rsi_100,dist_night_high_ticks,dist_prev_close_ticks
AUC loss,0.00525,0.00419,0.00394,0.00228,0.00203,0.00199,0.0018,0.00161,0.00155,0.00129,0.00121,0.00099,0.00095,0.00083,0.00078,0.00048,0.00045,0.00035,0.00024,0.00023,0.00009,0.00007,0.00004,0.0,0.0,0.0,0.0,0.0,-0.0,-0.00017,-0.00018,-0.00022,-0.00023,-0.00032,-0.00036,-0.0006,-0.00065,-0.00112


## 7. Trading Simulation and Threshold Selection (validation only)

One position at a time. At every candle: `P_long`, `P_short`; if the larger one is ≥ threshold, a trade in that direction is
opened at the close; its result and duration come from the barrier outcome of that candle. Net = gross − costs.
The threshold with the best **t-value** (Ø net ÷ std × √trades) and at least `MIN_VALIDATION_TRADES` trades is selected.

In [10]:
def simulate(outcome: dict, rows: np.ndarray, direction: np.ndarray) -> pd.DataFrame:
    """rows: candidate candles in time order, direction: +1 / -1 per candidate."""
    trades, next_free = [], -1
    for r, d in zip(rows, direction):
        if r < next_free:
            continue
        side = outcome["long"] if d == 1 else outcome["short"]
        trades.append({"t": int(r), "direction": int(d), "gross_ticks": float(side["result"][r]),
                       "exit": int(side["exit"][r]), "minutes": int(side["minutes"][r])})
        next_free = r + int(side["minutes"][r])
    return pd.DataFrame(trades)


def summarize(trades: pd.DataFrame, timestamps: pd.Series, days: int) -> dict:
    """timestamps: all candles of the data set (trade rows index into it), days: trading days of the period."""
    if trades.empty:
        return {"Trades": 0}
    net = trades["gross_ticks"].to_numpy() - COST_PER_TRADE_TICKS
    equity = np.concatenate([[0.0], np.cumsum(net * TICK_VALUE_USD)])
    drawdown = (np.maximum.accumulate(equity) - equity).max()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    ts = timestamps.iloc[trades["t"].to_numpy()]
    monthly = pd.Series(net, index=ts.dt.strftime("%Y-%m").to_numpy()).groupby(level=0).sum()
    days = max(days, 1)
    is_long = trades["direction"].to_numpy() == 1
    return {
        "Trades": len(trades), "Trades/day": len(trades) / days, "Long %": 100 * is_long.mean(),
        "Target %": 100 * (trades["exit"] == EXIT_TARGET).mean(),
        "Stop %": 100 * (trades["exit"] == EXIT_STOP).mean(),
        "Time exit %": 100 * (trades["exit"] == EXIT_TIME).mean(),
        "Ø net (ticks)": net.mean(),
        "Ø net long": net[is_long].mean() if is_long.any() else np.nan,
        "Ø net short": net[~is_long].mean() if (~is_long).any() else np.nan,
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Net total USD/contract": net.sum() * TICK_VALUE_USD,
        "Max drawdown USD": drawdown,
    }


def signals(p_long: np.ndarray, p_short: np.ndarray, threshold: float):
    best = np.maximum(p_long, p_short)
    mask = best >= threshold
    direction = np.where(p_long >= p_short, 1, -1)
    return mask, direction


probs = {"val": {}, "test": {}}
chosen = {}
for s in SETUPS:
    name = s["name"]
    o_tr = outcomes_train[name]
    va = val_rows[o_tr["valid"][val_rows]]
    p_l = models[name]["long"].predict_proba(X_train_all.iloc[va])[:, 1]
    p_s = models[name]["short"].predict_proba(X_train_all.iloc[va])[:, 1]
    probs["val"][name] = (va, p_l, p_s)

    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(o_tr, va[mask], direction[mask])
        rows.append({"Threshold": thr, "Signal candles": int(mask.sum()),
                     **summarize(tr, df_train["timestamp"], val_days)})
    val_table = pd.DataFrame(rows)
    eligible = val_table[val_table["Trades"] >= MIN_VALIDATION_TRADES]
    if eligible.empty or eligible["t-value"].isna().all():
        chosen[name] = None
        print(f"[{name}] no threshold with ≥ {MIN_VALIDATION_TRADES} validation trades – setup not tradable")
    else:
        best = eligible.loc[eligible["t-value"].idxmax()]
        chosen[name] = float(best["Threshold"])
        print(f"[{name}] chosen threshold {chosen[name]:.3f}: validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
              f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")
    print(f"[{name}] validation results per threshold:")
    display(val_table.round(3))

in] chosen threshold 0.525: validation Ø net -1.22 ticks, t-value -0.53, 176 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,164315,26453,61.519,40.321,35.697,36.416,27.887,-2.041,-1.564,-2.363,0.832,-12.025,0/17,-539830.0,546900.0
1,0.325,113626,20511,47.700,40.174,37.375,37.809,24.816,-2.003,-1.612,-2.266,0.839,-10.198,0/17,-410810.0,419070.0
2,0.350,73833,14971,34.816,40.158,38.875,39.122,22.003,-1.984,-1.617,-2.230,0.845,-8.427,0/17,-297060.0,302070.0
3,0.375,45455,10285,23.919,40.856,40.000,40.865,19.135,-1.996,-1.671,-2.220,0.849,-6.828,1/17,-205280.0,211400.0
4,0.400,26196,6611,15.374,42.823,42.021,42.218,15.762,-1.513,-1.284,-1.684,0.887,-4.054,2/17,-100020.0,115750.0
5,0.425,13519,3934,9.149,45.755,42.883,43.518,13.599,-2.003,-1.701,-2.258,0.857,-4.072,2/17,-78790.0,90070.0
6,0.450,6146,2069,4.812,45.771,45.288,42.533,12.180,-1.247,-0.498,-1.879,0.910,-1.820,6/17,-25800.0,40500.0
7,0.475,2568,968,2.251,47.417,44.525,43.802,11.674,-1.844,-0.340,-3.200,0.870,-1.877,7/17,-17850.0,26060.0
8,0.500,1121,427,0.993,52.225,43.794,44.262,11.944,-2.192,-2.408,-1.956,0.850,-1.494,5/17,-9360.0,14640.0
9,0.525,460,176,0.409,51.705,46.023,44.886,9.091,-1.216,-2.396,0.047,0.916,-0.534,7/15,-2140.0,5030.0


in] chosen threshold 0.600: validation Ø net -0.40 ticks, t-value -0.09, 143 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,200386,8055,18.733,35.270,35.854,35.593,28.554,-1.985,-1.075,-2.480,0.908,-3.525,3/17,-159870.0,174880.0
1,0.325,159870,6940,16.140,35.706,36.599,37.911,25.490,-2.882,-2.071,-3.333,0.869,-4.853,3/17,-200040.0,204280.0
2,0.350,122940,5789,13.463,35.291,37.450,38.625,23.925,-2.703,-1.884,-3.149,0.876,-4.229,2/17,-156470.0,159350.0
3,0.375,90673,4677,10.877,35.300,39.021,39.534,21.445,-2.413,-2.062,-2.604,0.888,-3.419,4/17,-112860.0,118900.0
4,0.400,64098,3589,8.347,34.968,40.457,40.346,19.198,-2.210,-1.760,-2.452,0.897,-2.757,4/17,-79310.0,90710.0
5,0.425,44127,2677,6.226,34.367,40.082,42.137,17.781,-3.000,-2.958,-3.022,0.864,-3.243,4/17,-80310.0,90550.0
6,0.450,29039,1904,4.428,35.032,39.916,42.332,17.752,-2.912,-3.715,-2.479,0.868,-2.640,2/17,-55450.0,70190.0
7,0.475,17966,1322,3.074,34.947,41.755,41.982,16.263,-2.328,-1.714,-2.658,0.893,-1.766,5/17,-30780.0,42370.0
8,0.500,11267,853,1.984,31.770,41.618,42.438,15.944,-2.546,-1.535,-3.017,0.888,-1.505,6/17,-21720.0,34760.0
9,0.525,6949,533,1.240,30.394,40.713,43.902,15.385,-3.677,1.006,-5.722,0.848,-1.663,8/17,-19600.0,29170.0


## 8. Test: Model vs. Random Direction vs. Always Long

Applied **once** with the threshold chosen on validation. The comparisons use exactly the same entry candles.

In [11]:
test_days = df_test["timestamp"].dt.date.nunique()
test_rows = []
test_trades = {}
for s in SETUPS:
    name = s["name"]
    o_te = outcomes_test[name]
    te = np.flatnonzero(o_te["valid"])
    p_l = models[name]["long"].predict_proba(X_test.iloc[te])[:, 1]
    p_s = models[name]["short"].predict_proba(X_test.iloc[te])[:, 1]
    probs["test"][name] = (te, p_l, p_s)
    thr = chosen[name]
    if thr is None:
        continue
    mask, direction = signals(p_l, p_s, thr)
    rows_sig = te[mask]
    strategies = {
        "Model": [direction[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=mask.sum()) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(mask.sum(), dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for d in dirs:
            tr = simulate(o_te, rows_sig, d)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Model":
                test_trades[name] = tr
        row = {"Setup": name, "Threshold": thr, "Strategy": strategy, "Signal candles": int(mask.sum())}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        test_rows.append(row)

test_df = pd.DataFrame(test_rows)
print(f"Rules: costs {COST_PER_TRADE_TICKS} ticks, target fill +{TARGET_FILL_EXTRA_TICKS} tick, stop first in the same "
      f"candle, no trades across pauses > {MAX_GAP_MINUTES} min, tick value {TICK_VALUE_USD} USD")
display(test_df.round(2))

Rules: costs 2.0 ticks, target fill +1 tick, stop first in the same candle, no trades across pauses > 5 min, tick value 10.0 USD


,Setup,Threshold,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,0.52,Model,437,193.0,0.62,54.40,41.45,42.49,16.06,2.28,7.93,-4.45,1.10,0.48,5/13,4410.0,6860.0
1,15min,0.52,Random direction (Ø 5),437,194.2,0.62,53.96,40.37,43.26,16.37,-3.99,0.82,-9.66,0.87,-0.85,7/13,-7754.0,13128.0
2,15min,0.52,Always long,437,194.0,0.62,100.00,43.30,40.21,16.49,2.48,2.48,NaN,1.10,0.52,7/13,4820.0,6870.0
3,60min,0.60,Model,1005,115.0,0.37,26.96,42.61,44.35,13.04,-2.32,1.48,-3.73,0.95,-0.22,7/13,-2670.0,14380.0
4,60min,0.60,Random direction (Ø 5),1005,115.0,0.37,49.04,42.26,44.70,13.04,-3.03,-0.79,-5.14,0.95,-0.29,6/13,-3490.0,13458.0
5,60min,0.60,Always long,1005,115.0,0.37,100.00,43.48,43.48,13.04,0.20,0.20,NaN,1.00,0.02,4/13,230.0,12330.0


### 8.1 For Information Only: All Thresholds on the Test Data

Do **not** pick the threshold from this table – that would be fitting to the test year. It only shows whether the
result is stable around the chosen threshold.

In [12]:
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(outcomes_test[name], te[mask], direction[mask])
        rows.append({"Threshold": thr, **summarize(tr, df_test["timestamp"], test_days)})
    print(f"[{name}] test results per threshold (chosen on validation: {chosen[name]})")
    display(pd.DataFrame(rows).round(3))

in] test results per threshold (chosen on validation: 0.525)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,20403,65.185,35.029,34.936,33.794,31.270,-1.147,-0.997,-1.229,0.957,-2.363,1/13,-234120.0,239050.0
1,0.325,16271,51.984,35.560,36.126,35.093,28.781,-1.672,-1.135,-1.968,0.937,-3.103,2/13,-271990.0,275650.0
2,0.350,12280,39.233,36.360,37.060,36.800,26.140,-2.022,-1.118,-2.539,0.926,-3.251,2/13,-248360.0,254920.0
3,0.375,8580,27.412,37.238,38.473,37.937,23.590,-1.703,0.018,-2.725,0.937,-2.293,2/13,-146160.0,153140.0
4,0.400,5611,17.927,39.904,40.349,38.585,21.066,-1.189,0.436,-2.267,0.957,-1.265,5/13,-66690.0,88660.0
5,0.425,3353,10.712,43.424,41.962,39.069,18.968,0.199,2.043,-1.217,1.007,0.165,6/13,6660.0,31670.0
6,0.450,1919,6.131,47.108,42.731,41.532,15.737,-0.224,2.775,-2.895,0.992,-0.140,7/13,-4290.0,28920.0
7,0.475,1001,3.198,49.151,41.858,43.157,14.985,-1.219,-0.626,-1.792,0.956,-0.531,5/13,-12200.0,22650.0
8,0.500,469,1.498,51.812,42.217,44.563,13.220,-1.141,-1.317,-0.951,0.958,-0.369,4/13,-5350.0,15930.0
9,0.525,193,0.617,54.404,41.451,42.487,16.062,2.285,7.933,-4.455,1.095,0.481,5/13,4410.0,6860.0


in] test results per threshold (chosen on validation: 0.6)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,5615,17.939,29.012,34.337,34.871,30.793,-2.771,-2.858,-2.736,0.942,-1.746,3/13,-155600.0,176240.0
1,0.325,4763,15.217,29.477,34.726,35.566,29.708,-3.391,-2.953,-3.575,0.928,-2.071,4/13,-161530.0,166020.0
2,0.350,3943,12.597,29.825,36.394,35.861,27.745,-2.587,-3.675,-2.124,0.943,-1.472,5/13,-102000.0,125750.0
3,0.375,3160,10.096,30.253,37.722,36.835,25.443,-2.693,-3.380,-2.396,0.940,-1.414,4/13,-85110.0,105960.0
4,0.400,2435,7.780,30.349,37.782,38.439,23.778,-3.652,-4.089,-3.461,0.920,-1.669,4/13,-88920.0,91450.0
5,0.425,1863,5.952,29.844,38.808,39.077,22.115,-2.213,-4.146,-1.391,0.950,-0.922,5/13,-41230.0,47650.0
6,0.450,1356,4.332,29.572,39.233,38.864,21.903,-1.185,-0.838,-1.331,0.973,-0.413,6/13,-16070.0,32080.0
7,0.475,978,3.125,28.016,39.264,40.184,20.552,-2.422,-0.350,-3.229,0.944,-0.750,6/13,-23690.0,49720.0
8,0.500,700,2.236,27.714,41.143,39.000,19.857,-0.309,7.577,-3.332,0.992,-0.081,8/13,-2160.0,27870.0
9,0.525,470,1.502,27.234,41.277,41.064,17.660,-1.121,6.305,-3.901,0.973,-0.247,7/13,-5270.0,25110.0


## 8.2 Direction Strategy: Direction Model + Timing

* **Direction:** short if `P(up first)` is in the lowest `X %`, long if it is in the highest `X %` (cut-offs from the
  validation distribution, `DIRECTION_QUANTILES`).
* **Timing:** additionally the win model of the traded side must be at least at its validation quantile
  `TIMING_QUANTILES` (0 = no filter) – i.e. a move large enough to reach the target is expected.
* **Sides:** both, short only, long only (`DIRECTION_SIDES`).
* Same exits as above (volatility target / stop, stop first in the same candle, +1 tick for the target, time exit),
  one position at a time, costs included.

All combinations are evaluated on the **validation period**; the best t-value (with at least `MIN_VALIDATION_TRADES`
trades) is applied **once** to the test data and compared with random direction, always long and always short on exactly
the same entry candles. Long and short are reported separately.

In [13]:
def direction_signals(p_up, p_long, p_short, cfg):
    long_ = (p_up >= cfg["p_up_high"]) & (p_long >= cfg["p_long_min"])
    short_ = (p_up <= cfg["p_up_low"]) & (p_short >= cfg["p_short_min"])
    if cfg["side"] == "short":
        long_[:] = False
    elif cfg["side"] == "long":
        short_[:] = False
    return long_ | short_, np.where(long_, 1, -1)


dir_chosen, dir_test_rows, dir_trades, dir_signal_rows = {}, [], {}, {}
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    pu_va = direction_models[name].predict_proba(X_train_all.iloc[va])[:, 1]
    pu_te = direction_models[name].predict_proba(X_test.iloc[te])[:, 1]
    pl_va, ps_va = probs["val"][name][1], probs["val"][name][2]
    pl_te, ps_te = probs["test"][name][1], probs["test"][name][2]
    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()

    # ---- grid on validation
    grid = []
    for q in DIRECTION_QUANTILES:
        for tq in TIMING_QUANTILES:
            for side in DIRECTION_SIDES:
                cfg = {
                    "direction_quantile": q, "timing_quantile": tq, "side": side,
                    "p_up_low": float(np.quantile(pu_va, q)), "p_up_high": float(np.quantile(pu_va, 1 - q)),
                    "p_long_min": float(np.quantile(pl_va, tq)) if tq > 0 else -np.inf,
                    "p_short_min": float(np.quantile(ps_va, tq)) if tq > 0 else -np.inf,
                }
                mask, d = direction_signals(pu_va, pl_va, ps_va, cfg)
                tr = simulate(o_tr, va[mask], d[mask])
                grid.append({**cfg, **summarize(tr, df_train["timestamp"], val_days)})
    grid_df = pd.DataFrame(grid)
    eligible = grid_df[(grid_df["Trades"] >= MIN_VALIDATION_TRADES) & grid_df["t-value"].notna()]
    print(f"[{name}] validation: all combinations")
    display(grid_df.drop(columns=["p_long_min", "p_short_min"]).round(3))
    if eligible.empty:
        print(f"[{name}] no combination with ≥ {MIN_VALIDATION_TRADES} validation trades")
        continue
    best = eligible.loc[eligible["t-value"].idxmax()]
    cfg = {k: best[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
    dir_chosen[name] = cfg
    print(f"[{name}] chosen on validation: side {cfg['side']}, direction {cfg['direction_quantile']:.0%}, "
          f"timing quantile {cfg['timing_quantile']} -> validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
          f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")

    # ---- test, applied once
    mask, d = direction_signals(pu_te, pl_te, ps_te, cfg)
    rows_sig = te[mask]
    dir_signal_rows[name] = (rows_sig, d[mask])
    n_sig = int(mask.sum())
    strategies = {
        "Direction model": [d[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=n_sig) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(n_sig, dtype=int)],
        "Always short": [-np.ones(n_sig, dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for dd in dirs:
            tr = simulate(o_te, rows_sig, dd)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Direction model":
                dir_trades[name] = tr
        row = {"Setup": name, "Side": cfg["side"], "Direction quantile": cfg["direction_quantile"],
               "Timing quantile": cfg["timing_quantile"], "Strategy": strategy, "Signal candles": n_sig}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        dir_test_rows.append(row)

    # ---- for information only: all combinations on the test data
    info = []
    for _, g in grid_df.iterrows():
        c = {k: g[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
        m2, d2 = direction_signals(pu_te, pl_te, ps_te, c)
        info.append({"direction_quantile": c["direction_quantile"], "timing_quantile": c["timing_quantile"],
                     "side": c["side"], **summarize(simulate(o_te, te[m2], d2[m2]), df_test["timestamp"], test_days)})
    dir_signal_rows[name + "_info"] = pd.DataFrame(info)

print("TEST – direction strategy with the combination chosen on validation:")
dir_test_df = pd.DataFrame(dir_test_rows)
display(dir_test_df.round(2))
for s in SETUPS:
    if s["name"] + "_info" in dir_signal_rows:
        print(f"[{s['name']}] for information only – all combinations on the test data (do not pick from here):")
        display(dir_signal_rows[s["name"] + "_info"].round(3))

in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.413,0.555,8922,20.749,54.270,29.186,26.989,43.824,-0.932,-0.221,-1.776,0.930,-2.529,3/17,-83160.0,99550.0
1,0.05,0.00,short,0.413,0.555,4104,9.544,0.000,27.729,25.804,46.467,-1.776,NaN,-1.776,0.893,-2.732,5/17,-72880.0,90040.0
2,0.05,0.00,long,0.413,0.555,4860,11.302,100.000,30.267,27.922,41.811,-0.244,-0.244,NaN,0.977,-0.617,5/17,-11860.0,29340.0
3,0.05,0.50,both,0.413,0.555,6552,15.237,55.907,33.455,30.617,35.928,-0.917,-0.285,-1.719,0.933,-2.133,3/17,-60100.0,68070.0
4,0.05,0.50,short,0.413,0.555,2894,6.730,0.000,32.205,30.028,37.768,-1.718,NaN,-1.718,0.902,-2.165,6/17,-49730.0,55710.0
5,0.05,0.50,long,0.413,0.555,3663,8.519,100.000,34.453,31.067,34.480,-0.285,-0.285,NaN,0.973,-0.640,4/17,-10430.0,26060.0
6,0.05,0.75,both,0.413,0.555,4597,10.691,56.907,35.741,33.522,30.737,-1.038,-0.472,-1.785,0.927,-1.988,4/17,-47710.0,55730.0
7,0.05,0.75,short,0.413,0.555,1981,4.607,0.000,34.780,31.954,33.266,-1.791,NaN,-1.791,0.900,-1.853,6/17,-35470.0,43950.0
8,0.05,0.75,long,0.413,0.555,2616,6.084,100.000,36.468,34.709,28.823,-0.472,-0.472,NaN,0.959,-0.853,7/17,-12340.0,19570.0
9,0.10,0.00,both,0.436,0.536,16886,39.270,54.169,29.551,28.177,42.272,-1.365,-0.746,-2.097,0.891,-5.620,0/17,-230530.0,247720.0


in] chosen on validation: side long, direction 5%, timing quantile 0.0 -> validation Ø net -0.24 ticks, t-value -0.62, 4860 trades
in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.325,0.593,2397,5.574,58.281,28.828,27.242,43.930,-1.892,0.339,-5.008,0.929,-1.390,7/17,-45350.0,64840.0
1,0.05,0.00,short,0.325,0.593,1002,2.330,0.000,27.545,26.248,46.208,-4.962,NaN,-4.962,0.856,-2.006,8/17,-49720.0,58120.0
2,0.05,0.00,long,0.325,0.593,1404,3.265,100.000,29.772,27.778,42.450,0.536,0.536,NaN,1.025,0.354,10/17,7520.0,25580.0
3,0.05,0.50,both,0.325,0.593,1811,4.212,58.034,31.530,30.039,38.432,-1.267,-0.055,-2.943,0.951,-0.856,7/17,-22950.0,49470.0
4,0.05,0.50,short,0.325,0.593,761,1.770,0.000,29.829,29.304,40.867,-3.050,NaN,-3.050,0.909,-1.095,7/17,-23210.0,36330.0
5,0.05,0.50,long,0.325,0.593,1052,2.447,100.000,32.795,30.608,36.597,0.051,0.051,NaN,1.003,0.033,8/17,540.0,19130.0
6,0.05,0.75,both,0.325,0.593,1199,2.788,58.048,35.196,32.444,32.360,-1.470,1.152,-5.097,0.940,-0.877,6/17,-17620.0,31010.0
7,0.05,0.75,short,0.325,0.593,504,1.172,0.000,31.746,31.548,36.706,-5.254,NaN,-5.254,0.842,-1.644,5/17,-26480.0,33050.0
8,0.05,0.75,long,0.325,0.593,696,1.619,100.000,37.644,33.190,29.167,1.152,1.152,NaN,1.062,0.667,9/17,8020.0,9920.0
9,0.10,0.00,both,0.375,0.558,4746,11.037,55.731,30.847,27.518,41.635,0.145,1.288,-1.295,1.006,0.163,9/17,6860.0,45870.0


in] chosen on validation: side long, direction 10%, timing quantile 0.0 -> validation Ø net +0.95 ticks, t-value 0.95, 2674 trades
TEST – direction strategy with the combination chosen on validation:


,Setup,Side,Direction quantile,Timing quantile,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,long,0.05,0.0,Direction model,27133,4721.0,15.08,100.00,28.43,28.89,42.68,-2.03,-2.03,NaN,0.93,-1.83,3/13,-95830.0,126460.0
1,15min,long,0.05,0.0,Random direction (Ø 5),27133,4722.0,15.09,49.77,28.50,28.79,42.71,-2.15,-1.91,-2.39,0.92,-1.94,2/13,-101718.0,125122.0
2,15min,long,0.05,0.0,Always long,27133,4721.0,15.08,100.00,28.43,28.89,42.68,-2.03,-2.03,NaN,0.93,-1.83,3/13,-95830.0,126460.0
3,15min,long,0.05,0.0,Always short,27133,4725.0,15.10,0.00,28.23,29.14,42.62,-2.49,NaN,-2.49,0.91,-2.24,1/13,-117490.0,121170.0
4,60min,long,0.10,0.0,Direction model,46535,2212.0,7.07,100.00,28.89,31.42,39.69,-5.10,-5.10,NaN,0.91,-1.52,5/13,-112830.0,140710.0
5,60min,long,0.10,0.0,Random direction (Ø 5),46535,2214.0,7.07,50.39,30.51,29.96,39.53,-0.14,-2.79,2.55,1.00,-0.04,7/13,-3100.0,94334.0
6,60min,long,0.10,0.0,Always long,46535,2212.0,7.07,100.00,28.89,31.42,39.69,-5.10,-5.10,NaN,0.91,-1.52,5/13,-112830.0,140710.0
7,60min,long,0.10,0.0,Always short,46535,2216.0,7.08,0.00,31.14,29.51,39.35,0.40,NaN,0.40,1.01,0.12,5/13,8850.0,76420.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,14016,44.780,33.540,29.395,27.483,43.122,-0.821,-1.947,-0.252,0.971,-1.317,3/13,-115040.0,148810.0
1,0.05,0.00,short,9338,29.834,0.000,29.835,26.804,43.360,-0.253,NaN,-0.253,0.991,-0.337,5/13,-23590.0,102980.0
2,0.05,0.00,long,4721,15.083,100.000,28.426,28.892,42.682,-2.030,-2.030,NaN,0.926,-1.829,3/13,-95830.0,126460.0
3,0.05,0.50,both,10502,33.553,33.736,32.137,30.385,37.479,-1.329,-1.399,-1.293,0.954,-1.834,2/13,-139530.0,151590.0
4,0.05,0.50,short,6965,22.252,0.000,32.376,29.892,37.732,-1.259,NaN,-1.259,0.957,-1.464,3/13,-87660.0,106350.0
5,0.05,0.50,long,3547,11.332,100.000,31.689,31.407,36.904,-1.414,-1.414,NaN,0.949,-1.066,3/13,-50160.0,66060.0
6,0.05,0.75,both,7333,23.428,35.224,34.256,32.524,33.220,-1.513,-1.781,-1.367,0.948,-1.742,4/13,-110940.0,124580.0
7,0.05,0.75,short,4752,15.182,0.000,34.680,32.302,33.018,-1.372,NaN,-1.372,0.953,-1.341,7/13,-65180.0,73680.0
8,0.05,0.75,long,2585,8.259,100.000,33.501,32.959,33.540,-1.791,-1.791,NaN,0.937,-1.125,6/13,-46310.0,65250.0
9,0.10,0.00,both,20879,66.706,35.471,29.513,28.488,41.999,-1.085,-1.718,-0.737,0.960,-2.165,2/13,-226500.0,235140.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,4278,13.668,35.344,29.383,29.593,41.024,-2.664,-4.251,-1.796,0.952,-1.187,6/13,-113960.0,144170.0
1,0.05,0.00,short,2772,8.856,0.000,30.195,28.608,41.198,-1.636,NaN,-1.636,0.969,-0.654,5/13,-45340.0,82960.0
2,0.05,0.00,long,1527,4.879,100.000,27.832,31.172,40.995,-4.344,-4.344,NaN,0.928,-0.996,7/13,-66330.0,82770.0
3,0.05,0.50,both,3145,10.048,33.418,31.383,30.906,37.711,-1.222,-2.672,-0.494,0.976,-0.514,5/13,-38430.0,51870.0
4,0.05,0.50,short,2094,6.690,0.000,32.283,30.325,37.393,-0.448,NaN,-0.448,0.991,-0.166,6/13,-9380.0,47360.0
5,0.05,0.50,long,1054,3.367,100.000,29.602,31.973,38.425,-2.693,-2.693,NaN,0.950,-0.578,6/13,-28380.0,49750.0
6,0.05,0.75,both,2038,6.511,33.268,33.562,33.170,33.268,-2.028,-0.547,-2.767,0.959,-0.743,5/13,-41340.0,59570.0
7,0.05,0.75,short,1361,4.348,0.000,34.313,33.358,32.329,-2.707,NaN,-2.707,0.946,-0.855,5/13,-36840.0,44760.0
8,0.05,0.75,long,681,2.176,100.000,32.159,33.040,34.802,-0.821,-0.821,NaN,0.983,-0.159,5/13,-5590.0,38920.0
9,0.10,0.00,both,6133,19.594,34.518,30.360,29.659,39.980,-2.001,-5.103,-0.366,0.963,-1.115,4/13,-122730.0,171250.0


## 9. Write Test CSV with Signals, Save Models and Log

In [14]:
# Output = original test CSV (all columns as text, timestamp unchanged) + probabilities and signals per setup
test_text = pd.read_csv(TEST_CSV_PATH, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
add = pd.DataFrame({"_bar": df_test["Bar"].to_numpy("int64")})
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    col_l, col_s, col_sig = f"P_long_{name}", f"P_short_{name}", f"Signal_{name}"
    add[col_l] = np.nan
    add[col_s] = np.nan
    add.loc[te, col_l] = np.round(p_l, 4)
    add.loc[te, col_s] = np.round(p_s, 4)
    add[col_sig] = 0
    if chosen[name] is not None:
        mask, direction = signals(p_l, p_s, chosen[name])
        add.loc[te[mask], col_sig] = direction[mask]
    add[f"Target_ticks_{name}"] = outcomes_test[name]["target_ticks"]
    add[f"Stop_ticks_{name}"] = outcomes_test[name]["stop_ticks"]
    add[f"P_up_first_{name}"] = np.nan
    add.loc[te, f"P_up_first_{name}"] = np.round(direction_models[name].predict_proba(X_test.iloc[te])[:, 1], 4)
    add[f"Trade_{name}"] = 0                         # trades actually taken (one position at a time)
    if name in test_trades and not test_trades[name].empty:
        add.loc[test_trades[name]["t"].to_numpy(), f"Trade_{name}"] = test_trades[name]["direction"].to_numpy()
    add[f"DirSignal_{name}"] = 0                     # direction strategy (section 8.2): signal / trades taken
    add[f"DirTrade_{name}"] = 0
    if name in dir_signal_rows:
        rows_sig, dirs = dir_signal_rows[name]
        add.loc[rows_sig, f"DirSignal_{name}"] = dirs
    if name in dir_trades and not dir_trades[name].empty:
        add.loc[dir_trades[name]["t"].to_numpy(), f"DirTrade_{name}"] = dir_trades[name]["direction"].to_numpy()

test_text["_bar"] = pd.to_numeric(test_text["Bar"]).astype("int64")
out = test_text.merge(add, on="_bar", how="left", validate="one_to_one").drop(columns="_bar")
out.to_csv(TEST_OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")
print(f"✓ Written: {TEST_OUTPUT_CSV_PATH} ({len(out):,} rows, new columns: {list(add.columns[1:])})")

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_rows = []
for s in SETUPS:
    name = s["name"]
    path = MODEL_DIR / f"barrier_{SYMBOL}_{name}_{run_ts}.joblib"
    joblib.dump({
        "models": models[name], "direction_model": direction_models.get(name),
        "direction_strategy": dir_chosen.get(name),
        "features": FEATURE_COLUMNS, "setup": s, "threshold": chosen[name],
        "vol_lookback": VOL_LOOKBACK, "min_barrier_ticks": MIN_BARRIER_TICKS,
        "trained_until": str(df_train["timestamp"].iloc[val_start - 1]), "input_columns": INPUT_COLUMNS,
        "use_time_features": USE_TIME_FEATURES, "tick_size": TICK_SIZE,
        "rsi_periods": RSI_PERIODS, "twap_windows": TWAP_WINDOWS, "vwap_windows": VWAP_WINDOWS,
        "session_start_hour_et": SESSION_START_HOUR_ET, "us_open_et": US_OPEN_ET, "opening_range_minutes": OPENING_RANGE_MINUTES,
    }, path)
    joblib.dump(joblib.load(path), MODEL_DIR / f"barrier_{SYMBOL}_{name}_latest.joblib")
    print(f"✓ Saved: {path.resolve()}")
    model_row = test_df[(test_df["Setup"] == name) & (test_df["Strategy"] == "Model")]
    auc = pd.DataFrame(auc_rows)
    log_rows.append({
        "run_timestamp": run_ts, "setup": name, "horizon": s["horizon"], "mode": s.get("mode", "fixed"),
        "target": s.get("target_ticks", s.get("target_sigma")), "stop": s.get("stop_ticks", s.get("stop_sigma")),
        "threshold": chosen[name],
        "auc_direction_val": next((r["AUC direction validation"] for r in direction_rows if r["Setup"] == name), None),
        "auc_direction_test": next((r["AUC direction test"] for r in direction_rows if r["Setup"] == name), None),
        "auc_val_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC validation"].iloc[0]),
        "auc_val_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC validation"].iloc[0]),
        "auc_test_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC test"].iloc[0]),
        "auc_test_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC test"].iloc[0]),
        **({k: model_row.iloc[0][k] for k in ["Trades", "Ø net (ticks)", "Profit factor", "t-value",
                                             "Months in profit", "Net total USD/contract", "Max drawdown USD"]}
           if len(model_row) else {}),
        **({f"dir_{k}": dir_chosen[name][k] for k in ["side", "direction_quantile", "timing_quantile"]}
           if name in dir_chosen else {}),
        **({f"dir_{k}": v for k, v in dir_test_df[(dir_test_df["Setup"] == name)
                                                    & (dir_test_df["Strategy"] == "Direction model")].iloc[0][
               ["Trades", "Ø net (ticks)", "Profit factor", "t-value", "Months in profit", "Net total USD/contract"]].items()}
           if len(dir_test_df) and ((dir_test_df["Setup"] == name) & (dir_test_df["Strategy"] == "Direction model")).any() else {}),
        "costs_ticks": COST_PER_TRADE_TICKS, "train_row_stride": TRAIN_ROW_STRIDE, "model_path": str(path.resolve()),
    })
log_df = pd.DataFrame(log_rows)
if LOG_PATH.exists():
    log_df = pd.concat([pd.read_csv(LOG_PATH, sep=";"), log_df], ignore_index=True)
log_df.to_csv(LOG_PATH, sep=";", index=False)
print(f"✓ Logged to {LOG_PATH.resolve()}")

✓ Written: E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026_barrier_session_signals.csv (354,649 rows, new columns: ['P_long_15min', 'P_short_15min', 'Signal_15min', 'Target_ticks_15min', 'Stop_ticks_15min', 'P_up_first_15min', 'Trade_15min', 'DirSignal_15min', 'DirTrade_15min', 'P_long_60min', 'P_short_60min', 'Signal_60min', 'Target_ticks_60min', 'Stop_ticks_60min', 'P_up_first_60min', 'Trade_60min', 'DirSignal_60min', 'DirTrade_60min'])
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_session\barrier_GC_15min_20261026_203404.joblib
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_session\barrier_GC_60min_20261026_203404.joblib
✓ Logged to E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_session\barrier_log.csv
